import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Configuration
folder = r'C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Final Testing (Revision)\Results'
sites = ['AF', 'BC', 'DOU', 'POI', 'PGL', 'ZAN', 'JIN', 'LOU']
colors = {'AF':'#E6A4A4', 'BC':'#B8E3D8', 'DOU':'#7CCBA2', 'JIN':'#B8E3D8', 'LOU':'#F6C49B', 'POI':'#BFD7EA', 'PGL':'#7FB3D5', 'ZAN':'#7FB3D5'}
kstex = {'POI':0.16, 'PGL':0.79, 'DOU':0.36, 'ZAN':0.88, 'JIN':0.28, 'LOU':0.17, 'AF':0.08, 'BC':0.34}

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'stix'

# Load observations
obs = pd.read_excel(os.path.join(folder, 'gcb15001-sup-0001-tables1.xlsx'), sheet_name='SOC')
obs.columns = obs.columns.str.strip()
obs['Study'] = obs['Study'].str.strip().str.upper()
obs = obs.set_index('Study')

# Calculate percentage changes
rows = []

for site in sites:
    duration = int(obs.loc[site, 'Study Duration'])
    di = float(obs.loc[site, 'AI'])
    df = pd.read_csv(os.path.join(folder, f'{site}_Test_Zr.csv')).sort_values('Years')
    df['Ksat'] = kstex[site] + df['ksstr']
    ct = df[df['Years'].between(520, 595)]
    row = {'Study':site, 'DI':di}

    for var in ['phi', 'Ksat']:
        ct_mean = ct[var].mean()
        nt_value = np.interp(600 + duration, df['Years'], df[var])
        row[var] = 100 * (nt_value - ct_mean) / ct_mean

    rows.append(row)

phys = pd.DataFrame(rows).sort_values('DI').reset_index(drop=True)
print(phys.round(2).to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
x = np.arange(len(phys))
offset = 0.12

for var, marker, shift in [('Ksat', '^', -offset), ('phi', 's', offset)]:
    xp = x + shift
    ax.plot(xp, phys[var], color='0.5', linewidth=1, zorder=1)

    for i, row in phys.iterrows():
        ax.scatter(xp[i], row[var], marker=marker, s=180, color=colors[row['Study']], edgecolor='black', zorder=3)

#ax.axhline(0, color='black', linewidth=0.7)
ax.set_xticks(x)
ax.set_xticklabels(phys['Study'], fontsize=30)
ax.set_ylabel('Change from CT to NT (%)', fontsize=30)
ax.tick_params(axis='both', labelsize=30)

handles = [
    Line2D([0], [0], marker='^', color='0.5', markerfacecolor='white', markeredgecolor='black', markersize=12, label=r'$K_{sat}$'),
    Line2D([0], [0], marker='s', color='0.5', markerfacecolor='white', markeredgecolor='black', markersize=12, label=r'$\phi$')
]

ax.legend(handles=handles, fontsize=30, frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()